# Projeto parcial de classificação do atraso na partida
**Disciplina:** Introdução ao Machine Learning — IDP  
**Docente:** Profa. Dra. Roberta Moreira Wichmann  
**Discentes:** Francisco Fonseca, Lucimar Oliveira do Nascimento e Osvaldo Souza  
**Representante:** Lucimar Oliveira do Nascimento

## 1. Problema, fonte e objetivo
**Pergunta de pesquisa:** Em que medida informações disponíveis antes da partida permitem classificar a faixa de atraso na partida dos voos realizados registrados na base VRA/ANAC de 2024 e 2025?

Trata-se de uma proposta de **aprendizado supervisionado de classificação multiclasse ordinal**. Esta Atividade 1 descreve os dados, constrói o alvo e examina sua qualidade. Treinamento e avaliação de modelos ficam para a Atividade 2; a análise exploratória não demonstra capacidade preditiva.

A fonte institucional é a ANAC, por meio dos registros VRA disponibilizados no SIROS. A entrada deste trabalho é um arquivo já consolidado, com campos de origem e campos derivados de preparação anterior. Ele não equivale a um arquivo mensal original intocado da ANAC. A unidade observacional é o registro de uma etapa de voo; sua unicidade é auditada. O recorte inclui registros de ligações internacionais presentes na base, não apenas voos domésticos.

**Execução:** mantenha este notebook e a base entregue na mesma pasta, ou execute-o a partir da raiz do repositório. A base pode permanecer no ZIP oficial do projeto; não é necessário extrair o CSV. O carregamento aceita somente os nomes explícitos documentados abaixo e nunca escolhe arbitrariamente outro CSV.

**Alvo principal:** `codigo_faixa_atraso`, representado pelo rótulo `faixa_atraso_partida`, deriva de `atraso_partida_minutos = (partida_real − partida_prevista)` em minutos:

| Código | Faixa | Intervalo em minutos |
| --- | --- | --- |
| 0 | Pontual ou antecipado | x ≤ 0 |
| 1 | Atraso inferior a 15 min | 0 < x < 15 |
| 2 | Atraso de 15 a 30 min | 15 ≤ x ≤ 30 |
| 3 | Atraso superior a 30 até 45 min | 30 < x ≤ 45 |
| 4 | Atraso superior a 45 até 60 min | 45 < x ≤ 60 |
| 5 | Atraso superior a 60 min | x > 60 |

As seis faixas são uma definição operacional deste estudo. Não representam, por si só, categorias legais de assistência ao passageiro. O alvo secundário `atraso_bi` vale 1 para x ≥ 15 e 0 para x < 15; a classe 0 binária inclui pequenos atrasos e não é idêntica à faixa 0 multiclasse. Cancelamentos constituem outro desfecho e ficam fora da população do alvo de atraso.


In [ ]:
# Bibliotecas e configurações de apresentação
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('default')
plt.rcParams.update({'figure.figsize': (11, 5), 'font.size': 10})
pd.set_option('display.max_columns', 40)
print('Python:', sys.version.split()[0], '| pandas:', pd.__version__, '| NumPy:', np.__version__)


In [ ]:
# Leitura determinística do arquivo de entrada deste trabalho
grupos_candidatos = [
    [
        Path('base_lucimar_oliveira_do_nascimento.csv'),
        Path('base_lucimar_oliveira_do_nascimento.zip'),
        Path('base_lucimar_nascimento_v2.csv'),
        Path('base_lucimar_nascimento_v2.zip'),
    ],
    [
        Path('data/raw/base_lucimar_nascimento_v2.zip'),
        Path('../../data/raw/base_lucimar_nascimento_v2.zip'),
    ],
]
caminho_base = None
for grupo in grupos_candidatos:
    encontrados = [caminho for caminho in grupo if caminho.is_file()]
    if len(encontrados) > 1:
        raise FileExistsError(
            'Há mais de uma base no mesmo nível de prioridade: '
            f'{[str(caminho) for caminho in encontrados]}'
        )
    if encontrados:
        caminho_base = encontrados[0]
        break
if caminho_base is None:
    raise FileNotFoundError(
        'Nenhuma base oficial foi encontrada entre os nomes e locais aceitos.'
    )

COLUNAS_CSV = ['companhia_icao', 'empresa', 'numero_voo', 'codigo_di', 'codigo_tipo_linha',
 'modelo_equipamento', 'numero_assentos', 'origem_icao', 'origem_descricao',
 'partida_prevista', 'partida_real', 'destino_icao', 'destino_descricao',
 'chegada_prevista', 'chegada_real', 'situacao_voo', 'situacao_partida',
 'situacao_chegada', 'codeshare', 'cancelado', 'realizado', 'atraso_partida_min',
 'atraso_chegada_min', 'arquivo_origem', 'linha_origem']
identificadores = ['companhia_icao', 'numero_voo', 'codigo_di', 'codigo_tipo_linha',
 'modelo_equipamento', 'origem_icao', 'destino_icao', 'codeshare']

df_bruto = pd.read_csv(
    caminho_base,
    sep=',',
    encoding='utf-8-sig',
    dtype={coluna: 'string' for coluna in identificadores},
    low_memory=False,
)
faltantes = set(COLUNAS_CSV) - set(df_bruto.columns)
extras = set(df_bruto.columns) - set(COLUNAS_CSV)
if faltantes or extras or len(df_bruto.columns) != len(COLUNAS_CSV):
    raise ValueError(f'Esquema inesperado. Faltam: {faltantes}; extras: {extras}')
if df_bruto.empty:
    raise ValueError('A base de entrada não possui registros.')

print('Arquivo de entrada:', caminho_base.name)
print(f'Dimensões: {len(df_bruto):,} linhas × {len(df_bruto.columns)} colunas.')
display(df_bruto.head(3))
display(pd.DataFrame({
    'Nulos na entrada': df_bruto.isna().sum(),
    '% nulos': df_bruto.isna().mean().mul(100).round(2),
}))
print(
    'Repetições de arquivo_origem + linha_origem:',
    int(df_bruto.duplicated(['arquivo_origem', 'linha_origem']).sum()),
)


## 2. Padronização do esquema

O mapeamento abaixo é específico para os 25 campos da base consolidada entregue. Os nomes da entrada são descritos na aba `Base consolidada` do dicionário; as variáveis selecionadas e derivadas para a análise constam em `Variáveis analíticas`. A renomeação ocorre apenas no DataFrame. Identificadores são preservados como texto, inclusive eventuais zeros iniciais. Colunas obrigatórias ausentes interrompem a execução, evitando gráficos vazios apresentados como análises concluídas.


In [ ]:
# Nomes existentes no CSV -> nomes usados na análise
MAPEAMENTO = {
    'companhia_icao': 'sg_empresa_icao',
    'numero_voo': 'nr_voo',
    'origem_icao': 'sg_icao_origem',
    'destino_icao': 'sg_icao_destino',
    'partida_prevista': 'dt_partida_prevista',
    'partida_real': 'dt_partida_real',
    'chegada_prevista': 'dt_chegada_prevista',
    'chegada_real': 'dt_chegada_real',
    'situacao_voo': 'cd_situacao_voo',
}
df_padronizado = df_bruto.rename(columns=MAPEAMENTO)
if df_padronizado.columns.duplicated().any():
    raise ValueError('Há nomes de colunas duplicados após o mapeamento.')

for coluna in ['sg_empresa_icao', 'sg_icao_origem', 'sg_icao_destino']:
    df_padronizado[coluna] = (
        df_padronizado[coluna].astype('string').str.strip().str.upper().replace('', pd.NA)
    )
df_padronizado['nr_voo'] = df_padronizado['nr_voo'].astype('string').str.strip().replace('', pd.NA)

print('Mapeamento aplicado:')
display(pd.DataFrame(MAPEAMENTO.items(), columns=['Coluna na entrada', 'Coluna na análise']))


## 3. População analítica, qualidade temporal e alvos

Retemos registros com situação `REALIZADO` (ou `REALIZADA`), horários de partida previsto e real interpretáveis e partida prevista nos anos de 2024 ou 2025. As exclusões são contadas em etapas, sem presumir que toda exclusão seja cancelamento. Datas inválidas tornam-se ausentes e são auditadas. Não se preenchem situação desconhecida ou datas ausentes com valores fictícios.

Segundo os metadados oficiais do VRA, horários previstos e realizados são publicados em horário de Brasília. A consolidação é conferida por comparação entre o atraso recalculado e o campo `atraso_partida_min`. Extremos são mantidos e sinalizados para investigação, sem corte arbitrário na população analítica.

A diferença entre chegada e partida previstas é mantida para diagnóstico com o nome legado `tempo_voo_planejado_minutos`. Valores negativos ou incompatíveis devem ser investigados. Dados disponíveis apenas depois do voo não podem ser preditores de um modelo pré-partida.


In [ ]:
# Reexecução desta célula sempre parte do DataFrame anterior aos filtros.
df_temporal = df_padronizado.copy()
cols_datas = ['dt_partida_prevista', 'dt_partida_real', 'dt_chegada_prevista', 'dt_chegada_real']
falhas_datas = []
for coluna in cols_datas:
    valores = df_temporal[coluna].astype('string').str.strip().replace('', pd.NA)
    convertidos = pd.to_datetime(valores, format='%Y-%m-%d %H:%M:%S', errors='coerce')
    falhas_datas.append({
        'Coluna': coluna,
        'Ausentes antes': int(valores.isna().sum()),
        'Falhas de conversão': int((valores.notna() & convertidos.isna()).sum()),
    })
    df_temporal[coluna] = convertidos
display(pd.DataFrame(falhas_datas))

situacao = df_temporal['cd_situacao_voo'].astype('string').str.strip().str.upper()
df_temporal['cd_situacao_voo'] = situacao
print('Situações registradas antes do filtro:')
display(situacao.fillna('(ausente)').value_counts().rename('Registros').to_frame())

realizados = situacao.isin(['REALIZADO', 'REALIZADA'])
partida_valida = df_temporal['dt_partida_prevista'].notna() & df_temporal['dt_partida_real'].notna()
periodo_valido = df_temporal['dt_partida_prevista'].dt.year.isin([2024, 2025])
mask_analise = realizados & partida_valida & periodo_valido
resumo_filtros = pd.DataFrame({
    'Etapa': [
        'Entrada',
        'Excluídos: situação diferente de realizado ou ausente',
        'Excluídos: realizados sem ambas as datas de partida válidas',
        'Excluídos: realizados com datas válidas fora de 2024–2025',
        'Retidos',
    ],
    'Registros': [
        len(df_temporal),
        int((~realizados).sum()),
        int((realizados & ~partida_valida).sum()),
        int((realizados & partida_valida & ~periodo_valido).sum()),
        int(mask_analise.sum()),
    ],
})
display(resumo_filtros)

df = df_temporal.loc[mask_analise].copy()
if df.empty:
    raise ValueError('Nenhum voo realizado de 2024–2025 possui datas de partida válidas.')

df['atraso_partida_minutos'] = (
    (df['dt_partida_real'] - df['dt_partida_prevista']).dt.total_seconds() / 60
)
FAIXAS = {
    0: 'Pontual ou antecipado',
    1: 'Atraso inferior a 15 min',
    2: 'Atraso de 15 a 30 min',
    3: 'Atraso superior a 30 até 45 min',
    4: 'Atraso superior a 45 até 60 min',
    5: 'Atraso superior a 60 min',
}

def classificar_faixas(valores):
    return np.select(
        [valores <= 0, valores < 15, valores <= 30, valores <= 45, valores <= 60],
        [0, 1, 2, 3, 4],
        default=5,
    ).astype('int64')

df['codigo_faixa_atraso'] = classificar_faixas(df['atraso_partida_minutos'])
df['faixa_atraso_partida'] = df['codigo_faixa_atraso'].map(FAIXAS)
df['atraso_bi'] = df['atraso_partida_minutos'].ge(15).astype('int64')

for nome, componente in [
    ('ano', 'year'),
    ('mes', 'month'),
    ('dia_semana', 'dayofweek'),
    ('hora_partida_prevista', 'hour'),
    ('minuto_partida_previsto', 'minute'),
]:
    df[nome] = getattr(df['dt_partida_prevista'].dt, componente)

df['tempo_voo_planejado_minutos'] = (
    (df['dt_chegada_prevista'] - df['dt_partida_prevista']).dt.total_seconds() / 60
)
df['rota'] = df['sg_icao_origem'] + '-' + df['sg_icao_destino']

print(f'Registros retidos: {len(df):,} de {len(df_bruto):,}.')
print('Período observado:', df['dt_partida_prevista'].min(), 'a', df['dt_partida_prevista'].max())

atraso_preexistente = pd.to_numeric(df['atraso_partida_min'], errors='coerce')
comparaveis = atraso_preexistente.notna()
divergentes = comparaveis & (atraso_preexistente - df['atraso_partida_minutos']).abs().gt(1e-6)
print('Atrasos comparáveis ao campo do CSV:', int(comparaveis.sum()), '| divergências:', int(divergentes.sum()))

suspeitos = df['atraso_partida_minutos'].lt(-60) | df['atraso_partida_minutos'].gt(1440)
print('Registros para inspeção (antecipação >60 min ou atraso >24h):', int(suspeitos.sum()))
campos_inspecao = [
    'arquivo_origem', 'linha_origem', 'sg_empresa_icao', 'nr_voo',
    'dt_partida_prevista', 'dt_partida_real', 'atraso_partida_minutos',
]
display(df.loc[suspeitos, campos_inspecao].head(20))

PREDITORES_CANDIDATOS = [
    'sg_empresa_icao', 'nr_voo', 'sg_icao_origem', 'sg_icao_destino',
    'rota', 'mes', 'dia_semana', 'hora_partida_prevista', 'minuto_partida_previsto',
]
print('Candidatos pré-partida para estudo posterior:', PREDITORES_CANDIDATOS)
print('Não usar como X: horários reais, situação final, atrasos calculados ou representações dos alvos.')


In [ ]:
# ==============================================================================
# CÉLULA 8: Medidas de Posição, Dispersão e Morfologia do Atraso na Partida
# ==============================================================================
percentis = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
desc = df['atraso_partida_minutos'].describe(percentiles=percentis)

tabela_estatisticas = pd.DataFrame({
    'Métrica Estatística': [
        'Contagem de Voos (N)', 'Média (min)', 'Desvio-Padrão (min)', 'Mínimo (min)',
        'Percentil 5% (min)', 'Quartil 1 - 25% (min)', 'Mediana - 50% (min)',
        'Quartil 3 - 75% (min)', 'Percentil 90% (min)', 'Percentil 95% (min)',
        'Percentil 99% (min)', 'Máximo (min)', 'Intervalo Interquartil - IQR (min)',
        'Assimetria (Skewness)', 'Curtose (Kurtosis)'
    ],
    'Valor Apurado': [
        f"{desc['count']:,.0f}",
        f"{desc['mean']:.2f}",
        f"{desc['std']:.2f}",
        f"{desc['min']:.2f}",
        f"{desc['5%']:.2f}",
        f"{desc['25%']:.2f}",
        f"{desc['50%']:.2f}",
        f"{desc['75%']:.2f}",
        f"{desc['90%']:.2f}",
        f"{desc['95%']:.2f}",
        f"{desc['99%']:.2f}",
        f"{desc['max']:.2f}",
        f"{(desc['75%'] - desc['25%']):.2f}",
        f"{df['atraso_partida_minutos'].skew():.2f}",
        f"{df['atraso_partida_minutos'].kurtosis():.2f}"
    ]
})

print("--- ESTATÍSTICAS DESCRITIVAS DE POSIÇÃO E DISPERSÃO (atraso_partida_minutos) ---")
display(tabela_estatisticas)

### Leitura das estatísticas

Valores negativos indicam antecipação no registro de partida; valores positivos indicam atraso. A mediana descreve a posição central com menor influência dos extremos do que a média. Assimetria e curtose elevadas exigem exame das caudas e da qualidade dos registros; não identificam, por si só, a causa dos atrasos.

Os valores extremos exibidos pela célula anterior devem ser conferidos por `arquivo_origem` e `linha_origem`. Eles permanecem na população analítica até que exista justificativa documentada para correção ou exclusão.


In [ ]:
# ==============================================================================
# CÉLULA 10: Distribuição de Frequência das 6 Faixas da Target e Target Binária
# ==============================================================================
ordem_codigos = [0, 1, 2, 3, 4, 5]
ordem_rotulos = [FAIXAS[k] for k in ordem_codigos]

contagem = df['codigo_faixa_atraso'].value_counts().reindex(ordem_codigos, fill_value=0)
proporcao = (contagem / len(df)) * 100

df_dist_target = pd.DataFrame({
    'Código': ordem_codigos,
    'Faixa de Atraso (faixa_atraso_partida)': ordem_rotulos,
    'Quantidade de Voos (N)': contagem.values,
    'Frequência Relativa (%)': proporcao.round(2).values,
    'Frequência Acumulada (%)': proporcao.cumsum().round(2).values
})

print("--- DISTRIBUIÇÃO DA VARIÁVEL TARGET PRINCIPAL (6 FAIXAS ORDINAIS) ---")
display(df_dist_target)

print("\n--- DISTRIBUIÇÃO DA VARIÁVEL TARGET SECUNDÁRIA (atraso_bi: >= 15 min) ---")
cont_bi = df['atraso_bi'].value_counts().reindex([0, 1], fill_value=0)
pct_bi = (cont_bi / len(df) * 100).round(2)
display(pd.DataFrame({
    'Classe Binária': ['0 - Abaixo de 15 min', '1 - Atrasado (>= 15 min)'],
    'Quantidade (N)': cont_bi.values,
    'Proporção (%)': pct_bi.values
}))

# Visualização Gráfica das 6 Faixas
fig, ax = plt.subplots(figsize=(11, 5))
cores_6_faixas = ['#2ca02c', '#66c2a5', '#ffd92f', '#fc8d62', '#e78ac3', '#d62728']

barras = ax.bar(ordem_rotulos, contagem.values, color=cores_6_faixas)

for p in ax.patches:
    altura = int(p.get_height())
    pct = (altura / len(df)) * 100
    ax.annotate(
        f'{altura:,}\n({pct:.1f}%)',
        (p.get_x() + p.get_width() / 2., altura),
        ha='center', va='bottom', fontsize=9, xytext=(0, 3), textcoords='offset points'
    )

ax.set_title("Distribuição de Frequência das 6 Faixas de Atraso na Partida (VRA/ANAC)", fontsize=13, fontweight='bold')
ax.set_xlabel("Faixa de Atraso na Partida (Ordem Operacional 0 a 5)", fontsize=11)
ax.set_ylabel("Quantidade de Voos", fontsize=11)
plt.xticks(rotation=20, ha='right')
ax.margins(y=0.15)
plt.tight_layout()
plt.show()


In [ ]:
# Recorte exclusivamente visual; df mantém todos os registros elegíveis.
# 
viz = df.loc[
    df["atraso_partida_minutos"].between(-30, 180),
    "atraso_partida_minutos"
]

print(
    f"No recorte visual: {len(viz):,}; "
    f"fora do recorte: {len(df) - len(viz):,}. "
    "Nenhuma linha removida de df."
)

if viz.empty:
    print("Não há observações no intervalo de −30 a 180 minutos.")
else:
    q1 = viz.quantile(0.25)
    mediana = viz.median()
    q3 = viz.quantile(0.75)

    fig, (ax_box, ax_hist) = plt.subplots(
        2, 1,
        figsize=(14, 8),
        gridspec_kw={"height_ratios": [0.42, 0.58]},
        constrained_layout=True
    )

    # Painel superior: aproximação para tornar a caixa legível
    ax_box.boxplot(
        viz,
        orientation="horizontal",
        widths=0.30,
        showfliers=False,
        patch_artist=True,
        boxprops={"facecolor": "#72B6A1", "edgecolor": "#254B45", "linewidth": 2},
        medianprops={"color": "#17252A", "linewidth": 3},
        whiskerprops={"color": "#254B45", "linewidth": 2},
        capprops={"color": "#254B45", "linewidth": 2}
    )

    ax_box.set_xlim(-30, 45)
    ax_box.set_ylim(0.65, 1.65)
    ax_box.set_yticks([])
    ax_box.set_title(
        "Onde se concentram os horários de partida",
        fontsize=15, fontweight="bold", loc="left", pad=15
    )
    ax_box.set_xlabel("Atraso na partida, em minutos — escala ampliada")
    ax_box.axvline(0, color="#338C62", linestyle="--", alpha=0.75)
    ax_box.axvline(15, color="#D98B18", linestyle="--", alpha=0.75)
    ax_box.text(q1, 1.34, f"Q1: {q1:.0f} min", ha="center", fontsize=10)
    ax_box.text(mediana, 1.52, f"Mediana: {mediana:.0f} min",
                ha="center", fontsize=10, fontweight="bold")
    ax_box.text(q3, 1.34, f"Q3: {q3:.0f} min", ha="center", fontsize=10)
    ax_box.grid(axis="x", alpha=0.2)
    ax_box.spines[["top", "right", "left"]].set_visible(False)

    # Painel inferior: distribuição em todo o recorte visual
    ax_hist.hist(
        viz, bins=60,
        color="#3576A3", edgecolor="white", linewidth=0.5
    )
    for valor, cor, texto in [
        (0, "#338C62", "Horário previsto: 0 min"),
        (15, "#D98B18", "Limiar do estudo: 15 min"),
        (60, "#C34B4B", "Limite da faixa 4: 60 min")
    ]:
        ax_hist.axvline(valor, color=cor, linestyle="--",
                        linewidth=2, label=texto)

    ax_hist.set_xlim(-30, 180)
    ax_hist.set_title(
        "Distribuição dos atrasos no recorte visual",
        fontsize=15, fontweight="bold", loc="left", pad=15
    )
    ax_hist.set_xlabel("Partida real menos partida prevista (minutos)")
    ax_hist.set_ylabel("Quantidade de voos")
    ax_hist.legend(frameon=False)
    ax_hist.grid(axis="y", alpha=0.2)
    ax_hist.spines[["top", "right"]].set_visible(False)

    plt.show()

In [ ]:
# Associações descritivas; diferenças entre grupos não estabelecem causalidade.
resumo_hora = df.groupby('hora_partida_prevista')['atraso_bi'].agg(N='size', Atrasados='sum', Proporcao='mean')
resumo_hora['Taxa (%)'] = resumo_hora.pop('Proporcao') * 100
resumo_cia = df.groupby('sg_empresa_icao')['atraso_bi'].agg(N='size', Atrasados='sum', Proporcao='mean')
resumo_cia['Taxa (%)'] = resumo_cia.pop('Proporcao') * 100
top5 = resumo_cia.sort_values('N', ascending=False).head(5)
print('Denominadores por hora prevista:')
display(resumo_hora)
print('Cinco companhias com maior número de registros nesta população:')
display(top5)
print('Registros sem sigla de companhia (fora do gráfico por companhia):', int(df['sg_empresa_icao'].isna().sum()))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
resumo_hora['Taxa (%)'].plot(marker='o', color='#b2182b', ax=axes[0])
axes[0].set(title='Atrasos ≥15 min por hora prevista', xlabel='Hora prevista de partida', ylabel='Taxa (%)')
axes[0].set_xticks(range(0, 24, 2))
if top5.empty:
    raise ValueError('Não há códigos de companhia válidos para o gráfico; verificar a coluna de origem.')
top5.sort_values('Taxa (%)', ascending=False)['Taxa (%)'].plot(kind='bar', color='#2b8cbe', ax=axes[1])
axes[1].set(title='Atrasos ≥15 min nas cinco companhias com mais registros', xlabel='Código ICAO', ylabel='Taxa (%)')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()
print('Cardinalidade das variáveis espaciais e companhia:')
display(df[['sg_empresa_icao','sg_icao_origem','sg_icao_destino','rota']].nunique().rename('Categorias distintas').to_frame())


## 4. Discussão preliminar

**População e variável de interesse.** O estudo examina a faixa de atraso na partida dos voos realizados de 2024 e 2025 com horários de partida previstos e reais válidos. Cancelamentos, datas inválidas e registros fora do período declarado são contabilizados separadamente. As tabelas anteriores mostram o desequilíbrio entre as seis classes; por isso, a etapa de modelagem deverá avaliar resultados por classe, macro-F1, acurácia balanceada e matriz de confusão.

**Descrição dos atrasos.** As medidas de posição e dispersão mostram a região mais frequente e a influência das caudas. O intervalo de −30 a 180 minutos usado no gráfico é exclusivamente visual e não remove observações da população analisada. Valores extremos precisam ser auditados antes de qualquer decisão de tratamento.

**Padrões exploratórios.** As taxas de atraso variam conforme a hora prevista e os grupos de companhias apresentados. Essas diferenças são associações descritivas, influenciadas também pela composição de rotas, aeroportos e períodos. Não permitem atribuir causalidade ao horário ou à companhia. Os denominadores devem acompanhar toda comparação.

**Qualidade e limites dos dados.** A interpretação depende da validade dos horários, da cobertura temporal e da consistência dos identificadores. O desfecho é o atraso na partida; atraso na chegada constitui outra pergunta de pesquisa. A análise exploratória descreve a população disponível e não mede capacidade de previsão.

**Próxima etapa.** A Atividade 2 deverá organizar treino, validação e teste em ordem temporal e usar como preditores apenas informações disponíveis no momento previsto para a estimativa. Horários reais, situação final, atrasos observados e variáveis derivadas do alvo não podem entrar na matriz preditora. Transformações que aprendam parâmetros deverão ser ajustadas somente no treino.
